In [1]:
import json
import os
import re

from num2words import num2words

In [2]:
_PUNCT_PATTERN = re.compile(r"[^\w\s]")
_WHITESPACE_PATTERN = re.compile(r"\s+")
_NUMBER_PATTERN = re.compile(r"\d+")


def remove_punctuation(text: str) -> str:
    return _PUNCT_PATTERN.sub(" ", text)


def normalize_numbers(text: str, lang: str = "id") -> str:
    return _NUMBER_PATTERN.sub(lambda m: num2words(int(m.group()), lang=lang), text)


def normalize(text: str) -> str:
    text = remove_punctuation(text)
    text = normalize_numbers(text)
    text = _WHITESPACE_PATTERN.sub(" ", text).strip().lower()
    return text

In [3]:
DATA_PATH = os.path.join("..", "data", "raw", "intent_dataset.jsonl")

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"loaded {len(records)} records from {DATA_PATH}")

loaded 2000 records from ../data/raw/intent_dataset.jsonl


In [4]:
for r in records:
    r["text_normalized"] = normalize(r["text"])

changed = [r for r in records if r["text"].strip().lower() != r["text_normalized"]]
print(f"{len(changed)} of {len(records)} rows changed by normalization\n")

for r in changed[:10]:
    print("RAW: ", r["text"])
    print("NORM:", r["text_normalized"])
    print()

1741 of 2000 rows changed by normalization

RAW:  eh, gue mau pesen nasi goreng spesial dua porsi
NORM: eh gue mau pesen nasi goreng spesial dua porsi

RAW:  anu, gue mau pesen ayam bakar satu ya bang
NORM: anu gue mau pesen ayam bakar satu ya bang

RAW:  bu, gue mau pesen mie ayam bakso dua mangkok
NORM: bu gue mau pesen mie ayam bakso dua mangkok

RAW:  um, gue pesen nasi goreng seafood satu porsi
NORM: um gue pesen nasi goreng seafood satu porsi

RAW:  mbak, gue mau pesen es teh manis tiga gelas
NORM: mbak gue mau pesen es teh manis tiga gelas

RAW:  hmm, gue pesen jus alpukat dua gelas ya pak
NORM: hmm gue pesen jus alpukat dua gelas ya pak

RAW:  mas, gue mau, mau pesen ayam goreng satu porsi
NORM: mas gue mau mau pesen ayam goreng satu porsi

RAW:  eh gue pesen mie goreng spesial dua, eh, tiga porsi pak
NORM: eh gue pesen mie goreng spesial dua eh tiga porsi pak

RAW:  na- nasi goreng ikan asin satu ya kak
NORM: na nasi goreng ikan asin satu ya kak

RAW:  bang gue mau pesen jus m

In [5]:
OUTPUT_PATH = os.path.join(
    "..", "data", "normalized", "intent_dataset_normalized.jsonl"
)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(records)} normalized records to {OUTPUT_PATH}")

wrote 2000 normalized records to ../data/normalized/intent_dataset_normalized.jsonl
